<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/Copy_of_HITL(LangSmith).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 22.1 MB/s eta 0:00:00


In [12]:
import json
import os
import uuid

from IPython.display import HTML
from google.colab import userdata
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Interrupt
from langsmith import Client as LangSmithClient
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_WORKSPACE_ID"] = userdata.get("LANGSMITH_WORKSPACE_ID")
os.environ["LANGSMITH_PROJECT"] = "Demo: Observability & LangSmith"

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


def print_interrupts(interrupts: List[Interrupt]):
    if not interrupts:
        print("There are no interrupts")
        return

    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [5]:
@tool
def search_travel_options(destination: str) -> str:
  """
  Call this tool to search for flight and accommodation options for a trip.
  """
  options = {
      "destination": destination,
      "flights": [
          {"flight_code": "LH170", "departure": "08:10", "price_eur": 189},
          {"flight_code": "FR402", "departure": "09:05", "price_eur": 129},
      ],
      "hotels": [
          { "hotel_name": "Asd Hub Hotel", "price_eur": 156 },
          { "hotel_name": "Spring View Stay", "price_eur": 144 }
      ]
  }

  return json.dumps(options, indent = 2)


@tool
def book_flight(traveler_name: str, flight_code: str) -> str:
  """
  Call this tool to book a flight.
  """
  return f"Booked flight {flight_code} for {traveler_name}"


@tool
def book_hotel(traveler_name: str, hotel_name: str, nights: int) -> str:
  """
  Call this tool to book a hotel.
  """
  return f"Booked {nights} night(s) at {hotel_name} for {traveler_name}"

за да използваме human-in-the-loop трябва да сетнем short-term-memory тъй като всеки един interrupt форсира прекратяването на работния цикъл на агента с което приключва текущия рън и когато предоставим approve, reject, edit, - тогава възобновяваме изпълнението, възобновяваме интеракцията с агнта само че вече в отделен рън, който има нужа от историята, какво се е случило.

In [9]:
agent = create_agent(
    model = ChatOpenAI(model="gpt-5-nano", api_key = openai_api_key, reasoning_effort = "low"),
    tools = [search_travel_options, book_flight, book_hotel],
    checkpointer = InMemorySaver(),
    middleware= [
        HumanInTheLoopMiddleware(
            interrupt_on ={
                book_flight.name: True,
                book_hotel.name: True, # за кои инструменти трябва да следи
            }
        )
    ]
)

In [10]:
run_id_1 = uuid.uuid4() # random unique id.

result_1 = agent.invoke(
    input = {
        "messages": [HumanMessage("Plan a one-night business trip to Berlin for Dana. Search options, book a morning flight, and reserve one hotel night for up to 150 EUR. If there are many possible options, take the initiative and select the most affordable one.")],
    },
    config = {
        "configurable": {
            "thread_id": "berlin_trip_3"
        },
        "run_id": run_id_1
    }
)

In [11]:
run_id_2 = uuid.uuid4() # random unique id.

result_2 = agent.invoke(
    input = Command(resume = {
        "decisions": [
            {"type": "approve"}, # за първия interrupt Tool: book_flight
            {"type": "approve"}, # за втория interrupt Tool: book_hotel
        ]
    }),
    config = {
        "configurable": {
            "thread_id": "berlin_trip_3"
        },
        "run_id": run_id_2
    }
)

In [13]:
lang_smith_client = LangSmithClient()

lang_smith_client.create_feedback(
    run_id = run_id_1,
    key = "user_feedback",
    score = 1.0
)

lang_smith_client.create_feedback(
    run_id = run_id_2,
    key = "user_feedback",
    score = 0.8
)

/tmp/ipykernel_2433/2162614739.py:3: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#feedback-create
  lang_smith_client.create_feedback(
/tmp/ipykernel_2433/2162614739.py:9: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#feedback-create
  lang_smith_client.create_feedback(


Feedback(id=UUID('01a03f1c-79f4-7860-b1c0-8f75e5f5fc49'), created_at=datetime.datetime(2026, 8, 26, 17, 27, 5, 972425, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 8, 26, 17, 27, 5, 972430, tzinfo=datetime.timezone.utc), run_id=UUID('959ea604-227f-4a7d-a3dc-e14b5f23ef72'), trace_id=None, key='user_feedback', score=0.8, value=None, comment=None, correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)